# RSNA Knee — submission (M1: frozen DINOv2-S + aggregator)

**Settings:** Accelerator **GPU T4** (not P100) · Internet **Off** · Inputs: the competition data and the private dataset `rsna-knee-models` (from `scripts/make_model_dataset.py`).

**Run All** first: with the 3 example test studies it finishes in a few minutes and writes `submission.csv`. Then **Save Version** (Save & Run All) and submit that version. During scoring Kaggle reruns it on the ~1,300 hidden test studies.

Pipeline: DICOM → `kneeprep` (CPU pool) → DINOv2-S slice features (GPU, fp16) → M1 fold ensemble with left/right-mirror TTA → `submission.csv`. A study that fails at any step gets the training prevalence, so the run never crashes. Generated by `scripts/build_submit_notebook.py` — edit the repo, not this notebook.

In [ ]:
import os, sys, subprocess, time
T_START = time.time()
TIME_LIMIT_H = 8.0          # stop embedding new studies after this; the rest get priors
WORKERS = os.cpu_count()    # CPU processes for DICOM decoding

In [ ]:
SKIP = {"train_series", "test_series", "train_images", "test_images"}

def find(predicate, top="/kaggle/input", max_depth=6):
    hits = []
    top_depth = top.rstrip("/").count("/")
    for root, dirs, files in os.walk(top):
        dirs[:] = [d for d in dirs if d not in SKIP]
        if root.count("/") - top_depth >= max_depth:
            dirs[:] = []
        if predicate(root, files):
            hits.append(root)
    return hits

DATA_ROOT = find(lambda r, f: "test_series.csv" in f and "sample_submission.csv" in f)
MODEL_DIR = find(lambda r, f: "priors.json" in f)
WHEELS = find(lambda r, f: any(x.endswith(".whl") for x in f))
assert DATA_ROOT, "competition data not found - Add Input -> Competitions"
assert MODEL_DIR, "model dataset not found - Add Input -> Datasets -> rsna-knee-models"
DATA_ROOT, MODEL_DIR = DATA_ROOT[0], MODEL_DIR[0]
print("data:", DATA_ROOT, "| models:", MODEL_DIR, "| wheels:", WHEELS[:1])
print("python", sys.version.split()[0], "| CPUs", os.cpu_count())
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])

In [ ]:
if WHEELS:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index",
                        "--find-links", WHEELS[0], "pydicom", "pylibjpeg", "pylibjpeg-libjpeg",
                        "pylibjpeg-openjpeg", "python-gdcm"], capture_output=True, text=True)
    print("wheel install:", "ok" if r.returncode == 0 else "FAILED\n" + r.stderr[-1500:])
else:
    print("no wheels found - using the preinstalled packages")
import pydicom, timm, torch
print("pydicom", pydicom.__version__, "| timm", timm.__version__, "| torch", torch.__version__)
for mod in ("pylibjpeg", "openjpeg", "libjpeg", "gdcm"):
    try:
        __import__(mod); print("  decoder available:", mod)
    except Exception:
        print("  decoder MISSING:", mod)

In [ ]:
EMPTY_FILES = ['src/__init__.py', 'src/models/__init__.py']
for d in ("src/kneeprep", "src/models"):
    os.makedirs(f"/kaggle/working/{d}", exist_ok=True)
for f in EMPTY_FILES:              # empty package markers (%%writefile needs a body)
    open(f"/kaggle/working/{f}", "w").close()

## Source files (embedded from the repo)

In [ ]:
%%writefile /kaggle/working/src/kneeprep/__init__.py
from .core import (  # noqa: F401
    CANON, KINDS, N_SLICES, PLANES, SIZE,
    build_volume, load_study, process_and_save, process_study, read_series, read_slice, select_series,
)


In [ ]:
%%writefile /kaggle/working/src/kneeprep/core.py
"""kneeprep: shared DICOM -> array preprocessing for RSNA Knee Abnormality Detection.

Used by BOTH the Kaggle prep notebook (training shards) and the submission notebook
(test studies), so train and test always go through identical preprocessing.
Edit this file only, then rebuild the notebooks (scripts/build_prep_notebook.py).

Output per series: uint8 volume [N_SLICES, SIZE, SIZE] in a canonical patient orientation
(DICOM patient coordinates are LPS: +x = patient left, +y = posterior, +z = superior):

  sagittal : slices patient right -> left | image up = superior | image right = posterior
  coronal  : slices anterior -> posterior | image up = superior | image right = patient left
  axial    : slices superior -> inferior  | image up = anterior | image right = patient left

Laterality is NOT normalized here (a left knee stays a left knee). Whether to mirror left
knees is a training-time choice; per-study laterality is recorded in the metadata.
"""
from __future__ import annotations

import os
from collections import Counter

import cv2
import numpy as np

N_SLICES = 32
SIZE = 256
PLANES = ("Sagittal", "Coronal", "Axial")

# plane -> ((image-right axis, sign), (image-down axis, sign), (slice axis, order sign)), LPS axes 0/1/2
CANON = {
    "Sagittal": ((1, +1), (2, -1), (0, +1)),
    "Coronal": ((0, +1), (2, -1), (1, +1)),
    "Axial": ((0, +1), (1, +1), (2, -1)),
}

# kind -> (plane, Fluid_Sensitive, preferred Fat_Suppression)
KINDS = {
    "sag_fl": ("Sagittal", 1, 1),
    "cor_fl": ("Coronal", 1, 1),
    "ax_fl": ("Axial", 1, 1),
    "sag_t1": ("Sagittal", 0, 0),
}

DUP_TOL_MM = 0.01  # slices closer than this along the normal count as the same position


# ----------------------------------------------------------------------------- DICOM I/O
def _get(ds, kw, default=None):
    v = ds.get(kw, None)
    if v is None or v == "":
        return default
    if hasattr(v, "__len__") and not isinstance(v, (str, bytes)) and kw in ("EchoNumbers",):
        return v[0] if len(v) else default
    return v


def _num(v, default=np.nan):
    try:
        return float(v)
    except (TypeError, ValueError):
        return default


def read_slice(path, with_pixels=True):
    """Read one DICOM file into a plain dict (geometry, a few tags, optional float32 pixels)."""
    import pydicom  # imported lazily so the geometry code can be tested without pydicom

    ds = pydicom.dcmread(path, force=True, stop_before_pixels=not with_pixels)
    iop = _get(ds, "ImageOrientationPatient")
    ipp = _get(ds, "ImagePositionPatient")
    sp = _get(ds, "PixelSpacing")
    lat = str(_get(ds, "Laterality", "") or _get(ds, "ImageLaterality", "") or "").strip().upper()[:1]
    rec = dict(
        iop=np.asarray(iop, float) if iop is not None and len(iop) == 6 else None,
        ipp=np.asarray(ipp, float) if ipp is not None and len(ipp) == 3 else None,
        spacing=(_num(sp[0], 1.0), _num(sp[1], 1.0)) if sp is not None and len(sp) == 2 else (1.0, 1.0),
        instance=int(_num(_get(ds, "InstanceNumber", 0), 0)),
        echo=int(_num(_get(ds, "EchoNumbers", 1), 1)),
        laterality=lat if lat in ("L", "R") else "",
        patient_id=str(_get(ds, "PatientID", "")),
        manufacturer=str(_get(ds, "Manufacturer", "")),
        field_strength=_num(_get(ds, "MagneticFieldStrength")),
        slice_thickness=_num(_get(ds, "SliceThickness")),
        spacing_between=_num(_get(ds, "SpacingBetweenSlices")),
    )
    if with_pixels:
        arr = ds.pixel_array
        if arr.ndim == 3:
            if int(_num(_get(ds, "SamplesPerPixel", 1), 1)) == 3:
                arr = arr.mean(-1)
            elif arr.shape[0] == 1:
                arr = arr[0]
            else:
                raise ValueError(f"multi-frame file with {arr.shape[0]} frames is not supported")
        arr = arr.astype(np.float32)
        slope = _num(_get(ds, "RescaleSlope", 1), 1.0)
        inter = _num(_get(ds, "RescaleIntercept", 0), 0.0)
        if slope != 1.0 or inter != 0.0:
            arr = arr * slope + inter
        if str(_get(ds, "PhotometricInterpretation", "")).upper() == "MONOCHROME1":
            arr = -arr  # inverted grayscale; the later percentile scaling handles the offset
        rec["pixels"] = arr
    return rec


# ----------------------------------------------------------------------------- geometry
def plane_of(iop):
    n = np.cross(iop[:3], iop[3:])
    return PLANES[int(np.argmax(np.abs(n)))]


def orient_ops(iop, plane):
    """(transpose, flip_lr, flip_ud) that bring a slice into the canonical orientation."""
    r, c = np.asarray(iop[:3], float), np.asarray(iop[3:], float)
    (ra, rs), (ca, cs), _ = CANON[plane]
    transpose = abs(c[ra]) > abs(r[ra])
    if transpose:
        r, c = c, r
    flip_lr = (np.sign(r[ra]) or 1.0) != rs
    flip_ud = (np.sign(c[ca]) or 1.0) != cs
    return bool(transpose), bool(flip_lr), bool(flip_ud)


def order_key(rec, plane):
    """Slice position along the plane's canonical slice axis (ascending = canonical order)."""
    _, _, (sa, so) = CANON[plane]
    n = np.cross(rec["iop"][:3], rec["iop"][3:])
    n = n / (np.linalg.norm(n) + 1e-12)
    s = np.sign(n[sa]) or 1.0
    return float(np.dot(rec["ipp"], n) * s * so)


def apply_ops(img, ops, spacing):
    t, lr, ud = ops
    rs, cs = spacing  # DICOM PixelSpacing = (row spacing, column spacing) in mm
    if t:
        img, rs, cs = img.T, cs, rs
    if lr:
        img = img[:, ::-1]
    if ud:
        img = img[::-1, :]
    return np.ascontiguousarray(img), (rs, cs)


def center_x(rec):
    """Patient x (left-right) of the slice centre; useful later for inferring laterality."""
    r, c = rec["iop"][:3], rec["iop"][3:]
    rows, cols = rec["pixels"].shape[:2]
    rs, cs = rec["spacing"]
    return float((rec["ipp"] + r * cs * (cols - 1) / 2 + c * rs * (rows - 1) / 2)[0])


# ----------------------------------------------------------------------------- image ops
def to_canvas(img, row_mm, col_mm, size=SIZE):
    """Resize keeping the physical aspect ratio, then centre-pad to size x size."""
    h, w = img.shape
    ph, pw = h * row_mm, w * col_mm
    scale = size / max(ph, pw)
    nh = int(min(size, max(1, round(ph * scale))))
    nw = int(min(size, max(1, round(pw * scale))))
    interp = cv2.INTER_AREA if (nh < h or nw < w) else cv2.INTER_LINEAR
    out = cv2.resize(img.astype(np.float32), (nw, nh), interpolation=interp)
    canvas = np.zeros((size, size), np.float32)
    y0, x0 = (size - nh) // 2, (size - nw) // 2
    canvas[y0:y0 + nh, x0:x0 + nw] = out
    return canvas, (y0, x0, nh, nw)


def resample_slices(vol, n=N_SLICES):
    """Resample the slice axis to n slices (block-average first when there are >= 2n slices)."""
    s = vol.shape[0]
    if s == n:
        return vol
    if s == 1:
        return np.repeat(vol, n, axis=0)
    k = s // n
    if k >= 2:
        start = (s - (s // k) * k) // 2
        vol = vol[start:start + (s // k) * k].reshape(-1, k, *vol.shape[1:]).mean(1)
        s = vol.shape[0]
        if s == n:
            return vol
    x = np.linspace(0, s - 1, n)
    i0 = np.floor(x).astype(int)
    i1 = np.minimum(i0 + 1, s - 1)
    w = (x - i0)[:, None, None].astype(np.float32)
    return vol[i0] * (1 - w) + vol[i1] * w


def to_uint8(vol, box, lo=0.5, hi=99.5):
    """Clip to the [lo, hi] percentiles of the image region (padding excluded) and scale to 0-255."""
    y0, x0, h, w = box
    roi = vol[:, y0:y0 + h, x0:x0 + w]
    a, b = np.percentile(roi[:, ::2, ::2], [lo, hi])
    if b <= a:
        b = a + 1.0
    out = np.zeros(vol.shape, np.uint8)
    out[:, y0:y0 + h, x0:x0 + w] = np.round(np.clip((roi - a) / (b - a), 0, 1) * 255).astype(np.uint8)
    return out


# ----------------------------------------------------------------------------- series / study
def build_volume(recs, n_slices=N_SLICES, size=SIZE):
    """Slice records (from read_slice) -> (uint8 [n_slices, size, size] volume, meta dict)."""
    recs = [r for r in recs if r is not None and r.get("pixels") is not None]
    if not recs:
        raise ValueError("no readable slices")
    mode = Counter(r["pixels"].shape for r in recs).most_common(1)[0][0]
    recs = [r for r in recs if r["pixels"].shape == mode]

    has_geo = all(r["iop"] is not None and r["ipp"] is not None for r in recs)
    ref = recs[len(recs) // 2]
    plane = plane_of(ref["iop"]) if ref["iop"] is not None else None
    if has_geo and plane is not None:
        keys, sorted_by = [order_key(r, plane) for r in recs], "position"
    else:
        keys, sorted_by = [float(r["instance"]) for r in recs], "instance"

    order = sorted(range(len(recs)), key=lambda i: (keys[i], recs[i]["echo"], recs[i]["instance"]))
    kept = []
    for i in order:
        if kept and sorted_by == "position" and abs(keys[i] - keys[kept[-1]]) < DUP_TOL_MM:
            continue  # same position (other echo / duplicate): keep the first
        kept.append(i)

    ops = orient_ops(ref["iop"], plane) if plane is not None else (False, False, False)
    slices, box = [], None
    for i in kept:
        img, sp = apply_ops(recs[i]["pixels"], ops, recs[i]["spacing"])
        can, box = to_canvas(img, sp[0], sp[1], size)
        slices.append(can)
    n_unique = len(slices)
    vol = to_uint8(resample_slices(np.stack(slices), n_slices), box)

    gaps = np.diff([keys[i] for i in kept]) if sorted_by == "position" and len(kept) > 1 else np.array([])
    meta = dict(
        plane_iop=plane or "", sorted_by=sorted_by, n_slices_used=len(recs), n_unique=n_unique,
        transposed=ops[0], flip_lr=ops[1], flip_ud=ops[2],
        slice_gap_mm=float(np.median(gaps)) if gaps.size else np.nan,
        rows=int(mode[0]), cols=int(mode[1]),
        row_spacing=float(ref["spacing"][0]), col_spacing=float(ref["spacing"][1]),
        slice_thickness=ref.get("slice_thickness", np.nan), spacing_between=ref.get("spacing_between", np.nan),
        fov_center_x=float(np.mean([center_x(recs[i]) for i in kept])) if has_geo else np.nan,
        laterality=ref.get("laterality", ""), patient_id=ref.get("patient_id", ""),
        manufacturer=ref.get("manufacturer", ""), field_strength=ref.get("field_strength", np.nan),
    )
    return vol, meta


def list_files(series_dir):
    return sorted(f for f in os.listdir(series_dir) if not f.startswith("."))


def read_series(series_dir, n_slices=N_SLICES, size=SIZE):
    files = list_files(series_dir)
    recs, n_failed = [], 0
    for f in files:
        try:
            recs.append(read_slice(os.path.join(series_dir, f)))
        except Exception:
            n_failed += 1
    vol, meta = build_volume(recs, n_slices, size)
    meta.update(n_files=len(files), n_failed=n_failed)
    return vol, meta


def select_series(study_rows, study_dir):
    """Pick one series per kind. study_rows: dicts with the train/test_series.csv columns."""
    cands = []
    for r in study_rows:
        d = os.path.join(study_dir, str(r["SeriesInstanceUID"]))
        n = len(list_files(d)) if os.path.isdir(d) else 0
        cands.append({**r, "n_files": n})
    picks = {}
    for kind, (plane, fluid, fs_pref) in KINDS.items():
        c = [r for r in cands
             if r["Anatomical_Plane"] == plane and int(r["Fluid_Sensitive"]) == fluid and r["n_files"] > 0]
        if c:
            picks[kind] = max(c, key=lambda r: (int(r["Fat_Suppression"]) == fs_pref, r["n_files"]))
    return picks, cands


def process_study(study_dir, study_rows, n_slices=N_SLICES, size=SIZE):
    """-> (arrays {kind: uint8 volume}, list of per-series meta dicts, study meta dict). Never raises
    for a single bad series: its error is recorded and the kind is left out of `arrays`."""
    study = os.path.basename(os.path.normpath(study_dir))
    picks, cands = select_series(study_rows, study_dir)
    arrays, series_meta = {}, []
    for kind, r in picks.items():
        sm = dict(StudyInstanceUID=study, kind=kind, SeriesInstanceUID=r["SeriesInstanceUID"],
                  Anatomical_Plane=r["Anatomical_Plane"], Fluid_Sensitive=int(r["Fluid_Sensitive"]),
                  Fat_Suppression=int(r["Fat_Suppression"]), error="")
        try:
            vol, m = read_series(os.path.join(study_dir, str(r["SeriesInstanceUID"])), n_slices, size)
            arrays[kind] = vol
            sm.update(m)
        except Exception as e:
            sm["error"] = f"{type(e).__name__}: {e}"[:300]
        series_meta.append(sm)

    # Study-level laterality and PatientID: kept series + one header from every other series
    lat, pids = Counter(), Counter()
    for m in series_meta:
        if m.get("laterality"):
            lat[m["laterality"]] += 1
        if m.get("patient_id"):
            pids[m["patient_id"]] += 1
    picked = {str(r["SeriesInstanceUID"]) for r in picks.values()}
    for r in cands:
        if str(r["SeriesInstanceUID"]) in picked or r["n_files"] == 0:
            continue
        d = os.path.join(study_dir, str(r["SeriesInstanceUID"]))
        try:
            h = read_slice(os.path.join(d, list_files(d)[0]), with_pixels=False)
            if h["laterality"]:
                lat[h["laterality"]] += 1
            if h["patient_id"]:
                pids[h["patient_id"]] += 1
        except Exception:
            pass
    laterality = ""
    if lat:
        top, cnt = lat.most_common(1)[0]
        laterality = top if cnt > sum(lat.values()) - cnt else ""  # strict majority only

    study_meta = dict(
        StudyInstanceUID=study,
        PatientID=pids.most_common(1)[0][0] if pids else "",
        laterality=laterality, lat_votes_L=lat["L"], lat_votes_R=lat["R"],
        kinds=",".join(k for k in KINDS if k in arrays),
        n_series_csv=len(cands), n_series_errors=sum(bool(m["error"]) for m in series_meta),
    )
    return arrays, series_meta, study_meta


def process_and_save(task):
    """Multiprocessing worker. task = (study_dir, study_rows, out_npz_path). Never raises."""
    import time

    study_dir, study_rows, out_path = task
    t0 = time.time()
    study = os.path.basename(os.path.normpath(study_dir))
    try:
        arrays, series_meta, study_meta = process_study(study_dir, study_rows)
        if arrays:
            tmp = out_path + ".tmp"
            with open(tmp, "wb") as fh:
                np.savez_compressed(fh, **arrays)
            os.replace(tmp, out_path)
        study_meta["npz_bytes"] = os.path.getsize(out_path) if arrays else 0
        study_meta["error"] = "" if arrays else "no series could be processed"
    except Exception as e:
        series_meta = []
        study_meta = dict(StudyInstanceUID=study, npz_bytes=0, error=f"{type(e).__name__}: {e}"[:300])
    study_meta["seconds"] = round(time.time() - t0, 2)
    return study_meta, series_meta


def load_study(npz_path):
    """-> {kind: uint8 [N_SLICES, SIZE, SIZE]} for the kinds present in the study."""
    with np.load(npz_path) as z:
        return {k: z[k] for k in z.files}


In [ ]:
%%writefile /kaggle/working/src/models/data.py
"""Pack the per-study DINOv2 embeddings into a few arrays for fast training.

    python -m src.models.data            # builds data/embeddings/<tag>/cache_g2/ once

Cache layout (all rows in the order of uids.txt):
    feats.npy    float16 [N, 6, 32, 768]        CLS + mean patch per slice
    grid.npy     float16 [N, 6, 32, g*g, 384]   patch grid pooled 4x4 -> g x g (g=2 default)
    present.npy  bool    [N, 6]                 which slots exist for the study
Slots: SLOTS below. Mirrored slots (cor_fl_m, ax_fl_m) hold the left-right flipped
images; sagittal kinds are mirrored by reversing the slice order instead.
"""
from __future__ import annotations

import argparse
import time
from pathlib import Path

import numpy as np

SLOTS = ("sag_fl", "cor_fl", "cor_fl_m", "ax_fl", "ax_fl_m", "sag_t1")
KINDS = ("sag_fl", "cor_fl", "ax_fl", "sag_t1")
# slot index of each kind, normal and mirrored
KIND_SLOT = np.array([0, 1, 3, 5])
KIND_SLOT_MIRROR = np.array([0, 2, 4, 5])
SAGITTAL = np.array([True, False, False, True])      # mirrored by reversing slice order
N_SLICES, FEAT_DIM, PATCH_DIM = 32, 768, 384


def pool_grid(grid: np.ndarray, g_out: int) -> np.ndarray:
    """[S, G*G, D] (row-major G x G) -> [S, g_out*g_out, D] by block averaging."""
    s, n, d = grid.shape
    g_in = int(round(n ** 0.5))
    if g_out == g_in:
        return grid
    f = g_in // g_out
    assert f * g_out == g_in, f"cannot pool {g_in}x{g_in} to {g_out}x{g_out}"
    x = grid.astype(np.float32).reshape(s, g_out, f, g_out, f, d).mean((2, 4))
    return x.reshape(s, g_out * g_out, d)


def build_cache(emb_dir: Path, g_out: int = 2) -> Path:
    files = sorted(p for p in emb_dir.glob("*.npz"))
    if not files:
        raise SystemExit(f"no embeddings in {emb_dir}")
    out = emb_dir / f"cache_g{g_out}"
    out.mkdir(exist_ok=True)
    n = len(files)
    feats = np.lib.format.open_memmap(out / "feats.tmp.npy", "w+", np.float16,
                                      (n, len(SLOTS), N_SLICES, FEAT_DIM))
    grid = np.lib.format.open_memmap(out / "grid.tmp.npy", "w+", np.float16,
                                     (n, len(SLOTS), N_SLICES, g_out * g_out, PATCH_DIM))
    present = np.zeros((n, len(SLOTS)), bool)
    t0 = time.time()
    for i, f in enumerate(files):
        with np.load(f) as z:
            for j, slot in enumerate(SLOTS):
                if slot in z.files:
                    feats[i, j] = z[slot]
                    grid[i, j] = pool_grid(z[f"{slot}_grid"], g_out)
                    present[i, j] = True
                else:
                    feats[i, j] = 0
                    grid[i, j] = 0
        if (i + 1) % 500 == 0 or i + 1 == n:
            print(f"  {i + 1}/{n}  {time.time() - t0:.0f}s")
    feats.flush()
    grid.flush()
    del feats, grid
    (out / "feats.tmp.npy").replace(out / "feats.npy")
    (out / "grid.tmp.npy").replace(out / "grid.npy")
    np.save(out / "present.npy", present)
    (out / "uids.txt").write_text("\n".join(f.stem for f in files))
    print(f"cache: {n} studies -> {out}")
    print("slots present:", dict(zip(SLOTS, present.sum(0).tolist())))
    return out


def channel_stats(cache: dict, chunk: int = 64) -> dict[str, np.ndarray]:
    """Per-channel mean / std over every present slice (no labels involved).

    feats: 768 channels; grid: 384 channels shared by all grid cells. DINOv2 has a few
    channels with very large values; standardising each channel stops them dominating.
    """
    feats, grid, present = cache["feats"], cache["grid"], cache["present"]
    sums = {k: np.zeros(d, np.float64) for k, d in (("f", feats.shape[-1]), ("g", grid.shape[-1]))}
    sq = {k: np.zeros_like(v) for k, v in sums.items()}
    n = {"f": 0, "g": 0}
    for s in range(0, len(feats), chunk):
        m = present[s:s + chunk]
        f = feats[s:s + chunk][m].astype(np.float64).reshape(-1, feats.shape[-1])
        g = grid[s:s + chunk][m].astype(np.float64).reshape(-1, grid.shape[-1])
        for key, a in (("f", f), ("g", g)):
            sums[key] += a.sum(0)
            sq[key] += (a * a).sum(0)
            n[key] += len(a)
    out = {}
    for key in ("f", "g"):
        mu = sums[key] / n[key]
        sd = np.sqrt(np.maximum(sq[key] / n[key] - mu * mu, 1e-12))
        out[f"{key}_mean"] = mu.astype(np.float32)
        out[f"{key}_std"] = np.maximum(sd, 1e-3).astype(np.float32)
    return out


def load_stats(cache_dir: Path, cache: dict) -> dict[str, np.ndarray]:
    path = cache_dir / "stats.npz"
    if path.exists():
        with np.load(path) as z:
            return {k: z[k] for k in z.files}
    stats = channel_stats(cache)
    np.savez(path, **stats)
    return stats


def load_cache(cache_dir: Path, in_memory: bool = True) -> dict:
    mode = None if in_memory else "r"
    return dict(
        uids=cache_dir.joinpath("uids.txt").read_text().split("\n"),
        feats=np.load(cache_dir / "feats.npy", mmap_mode=mode),
        grid=np.load(cache_dir / "grid.npy", mmap_mode=mode),
        present=np.load(cache_dir / "present.npy"),
    )


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", type=Path, default=Path("."))
    ap.add_argument("--tag", default="dinov2s_224_g4")
    ap.add_argument("--grid", type=int, default=2)
    args = ap.parse_args()
    build_cache(args.root / "data" / "embeddings" / args.tag, args.grid)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/src/models/embed.py
"""DINOv2-S slice embeddings for every study shard (frozen backbone, computed once).

Run from the project root (venv active):
    python -m src.models.embed --check      # 3 studies: shapes, sanity, speed, ETA
    python -m src.models.embed              # all studies; resumable (skips finished ones)

Input : data/shards/<StudyInstanceUID>.npz   {kind: uint8 [32, 256, 256]} (kneeprep)
Output: data/embeddings/<tag>/<StudyInstanceUID>.npz, for each kind present
        (sag_fl, cor_fl, ax_fl, sag_t1):
            <kind>        float16 [32, 768]          CLS (384) + mean patch token (384)
            <kind>_grid   float16 [32, G*G, 384]     patch tokens average-pooled to G x G
        and for the coronal/axial kinds also the LEFT-RIGHT MIRRORED images:
            <kind>_m, <kind>_m_grid
        plus data/embeddings/<tag>/config.json and index.csv.

Mirroring a knee (left <-> right) in kneeprep's standard orientation:
    coronal, axial : flip the image horizontally  -> needs the extra *_m features
    sagittal       : reverse the slice order       -> free at training time
So M1 can use "mirror the whole knee" as augmentation / test-time augmentation.

Inference uses fp16 autocast (the Kaggle T4 has no bf16), images are resized
256 -> 224 (whole field of view kept, no crop), grey -> 3 channels, ImageNet norm.
The same `Embedder` class must be used in the Kaggle submission notebook.
"""
from __future__ import annotations

import argparse
import csv
import json
import logging
import os
import time
from pathlib import Path

import numpy as np

KINDS = ("sag_fl", "cor_fl", "ax_fl", "sag_t1")
MIRROR_KINDS = ("cor_fl", "ax_fl")       # mirrored by a horizontal image flip
N_SLICES, SIZE = 32, 256
DEFAULT_MODEL = "vit_small_patch14_dinov2.lvd142m"


# ----------------------------------------------------------------------------- data side (numpy only)
def load_shard(path: Path) -> dict[str, np.ndarray]:
    """{kind: uint8 [32, 256, 256]}; raises ValueError on a malformed shard."""
    with np.load(path) as z:
        arrays = {k: z[k] for k in z.files}
    for k, a in arrays.items():
        if k not in KINDS:
            raise ValueError(f"unexpected key {k!r}")
        if a.dtype != np.uint8 or a.shape != (N_SLICES, SIZE, SIZE):
            raise ValueError(f"{k}: {a.dtype} {a.shape}")
    return arrays


def plan_images(arrays: dict[str, np.ndarray]) -> tuple[np.ndarray, list[tuple[str, int, int]]]:
    """Stack every image to embed into one uint8 [N, 256, 256] batch.

    Returns (images, segments) where segments = [(output_key, start, stop)].
    Mirrored kinds are appended as horizontally flipped copies.
    """
    blocks, segments, pos = [], [], 0
    for kind in KINDS:
        if kind not in arrays:
            continue
        variants = [(kind, arrays[kind])]
        if kind in MIRROR_KINDS:
            variants.append((f"{kind}_m", arrays[kind][:, :, ::-1]))
        for key, vol in variants:
            blocks.append(vol)
            segments.append((key, pos, pos + len(vol)))
            pos += len(vol)
    if not blocks:
        return np.zeros((0, SIZE, SIZE), np.uint8), []
    return np.ascontiguousarray(np.concatenate(blocks)), segments


def split_outputs(feat: np.ndarray, grid: np.ndarray,
                  segments: list[tuple[str, int, int]]) -> dict[str, np.ndarray]:
    out = {}
    for key, a, b in segments:
        out[key] = feat[a:b].astype(np.float16)
        out[f"{key}_grid"] = grid[a:b].astype(np.float16)
    return out


def save_atomic(path: Path, arrays: dict[str, np.ndarray]) -> None:
    tmp = path.with_suffix(".tmp")
    with open(tmp, "wb") as fh:
        np.savez(fh, **arrays)
    os.replace(tmp, path)


def cosine_rows(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    return (a * b).sum(-1) / (np.linalg.norm(a, axis=-1) * np.linalg.norm(b, axis=-1) + 1e-8)


def sanity_report(outputs: dict[str, dict[str, np.ndarray]], log: logging.Logger) -> bool:
    """Checks on a few embedded studies. Returns True if everything looks right."""
    ok = True
    uids = list(outputs)
    for uid in uids:
        o = outputs[uid]
        shapes = ", ".join(f"{k}{tuple(v.shape)}" for k, v in o.items() if not k.endswith("_grid"))
        log.info("  %s: %s", uid[-8:], shapes)
        for k, v in o.items():
            if not np.isfinite(v.astype(np.float32)).all():
                log.error("  NaN/inf in %s %s", uid[-8:], k)
                ok = False
    first = outputs[uids[0]]
    kind = next(k for k in KINDS if k in first)
    v = first[kind]
    adj = cosine_rows(v[:-1], v[1:]).mean()
    log.info("  cosine, neighbouring slices (%s): %.3f   [expect high, ~0.7-0.98]", kind, adj)
    for other_uid in uids[1:]:
        common = [k for k in KINDS if k in first and k in outputs[other_uid]]
        if common:
            k = common[0]
            cross = cosine_rows(first[k], outputs[other_uid][k]).mean()
            log.info("  cosine, %s same slice index, different study: %.3f   "
                     "[expect lower than neighbouring slices]", k, cross)
            break
    for k in MIRROR_KINDS:
        if k in first:
            c = cosine_rows(first[k], first[f"{k}_m"]).mean()
            log.info("  cosine, %s vs its mirror: %.3f   [expect < 0.999]", k, c)
            if c > 0.9995:
                log.error("  mirrored features identical to originals - flip not applied?")
                ok = False
    spread = float(np.std(v.astype(np.float32)))
    if spread < 1e-3:
        log.error("  features are (almost) constant - model not loaded correctly?")
        ok = False
    return ok


# ----------------------------------------------------------------------------- model side (torch)
class Embedder:
    """Frozen DINOv2 feature extractor. Same code for training features and Kaggle inference."""

    def __init__(self, model_name: str = DEFAULT_MODEL, img_size: int = 224, grid: int = 4,
                 weights: str | None = None, device: str = "cuda", batch_size: int = 256):
        import timm
        import torch

        self.torch = torch
        self.device = torch.device(device if torch.cuda.is_available() or device == "cpu" else "cpu")
        kw = dict(pretrained=True, num_classes=0, img_size=img_size)
        if weights:  # local weights file (Kaggle, internet off)
            kw["pretrained_cfg_overlay"] = dict(file=weights)
        self.model = timm.create_model(model_name, **kw).eval().to(self.device)
        cfg = self.model.pretrained_cfg
        self.mean = torch.tensor(cfg["mean"], device=self.device).view(1, 3, 1, 1)
        self.std = torch.tensor(cfg["std"], device=self.device).view(1, 3, 1, 1)
        self.img_size, self.grid, self.batch_size = img_size, grid, batch_size
        self.n_prefix = getattr(self.model, "num_prefix_tokens", 1)
        self.config = dict(model=model_name, img_size=img_size, grid=grid,
                           mean=list(cfg["mean"]), std=list(cfg["std"]),
                           n_prefix=self.n_prefix, embed_dim=self.model.num_features,
                           timm=timm.__version__, torch=torch.__version__, amp="fp16")

    def __call__(self, images: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """uint8 [N, H, W] -> (float32 [N, 2D], float32 [N, G*G, D])."""
        torch = self.torch
        F = torch.nn.functional
        feats, grids = [], []
        with torch.inference_mode():
            for s in range(0, len(images), self.batch_size):
                x = torch.from_numpy(images[s:s + self.batch_size]).to(self.device, non_blocking=True)
                x = x.float().div_(255.0).unsqueeze(1)                       # [B,1,H,W]
                if x.shape[-1] != self.img_size or x.shape[-2] != self.img_size:
                    x = F.interpolate(x, size=(self.img_size, self.img_size), mode="bilinear",
                                      align_corners=False, antialias=True)
                x = (x.expand(-1, 3, -1, -1) - self.mean) / self.std
                with torch.autocast(device_type=self.device.type, dtype=torch.float16,
                                    enabled=self.device.type == "cuda"):
                    tok = self.model.forward_features(x)                     # [B, P+L, D]
                tok = tok.float()
                cls = tok[:, 0]
                patches = tok[:, self.n_prefix:]
                b, n, d = patches.shape
                side = int(round(n ** 0.5))
                assert side * side == n, f"patch tokens {n} not a square grid"
                g = patches.transpose(1, 2).reshape(b, d, side, side)
                g = F.adaptive_avg_pool2d(g, self.grid).flatten(2).transpose(1, 2)  # [B,G*G,D]
                feats.append(torch.cat([cls, patches.mean(1)], 1).cpu().numpy())
                grids.append(g.cpu().numpy())
        return np.concatenate(feats), np.concatenate(grids)


# ----------------------------------------------------------------------------- loader (top-level for Windows spawn)
class ShardDataset:
    def __init__(self, paths):
        self.paths = list(paths)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        p = self.paths[i]
        try:
            images, segments = plan_images(load_shard(p))
            return p.stem, images, segments, ""
        except Exception as e:  # noqa: BLE001 - reported per study, run continues
            return p.stem, None, None, f"{type(e).__name__}: {e}"[:300]


def first_item(batch):
    return batch[0]


def iterate(dataset: ShardDataset, workers: int):
    if workers <= 0:
        for i in range(len(dataset)):
            yield dataset[i]
        return
    from torch.utils.data import DataLoader
    yield from DataLoader(dataset, batch_size=1, num_workers=workers, collate_fn=first_item,
                          prefetch_factor=4, persistent_workers=False)


# ----------------------------------------------------------------------------- main
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", type=Path, default=Path("."))
    ap.add_argument("--model", default=DEFAULT_MODEL)
    ap.add_argument("--weights", default=None, help="local weights file instead of downloading")
    ap.add_argument("--img-size", type=int, default=224)
    ap.add_argument("--grid", type=int, default=4)
    ap.add_argument("--batch-size", type=int, default=256)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--check", action="store_true", help="3 studies + sanity checks + ETA")
    ap.add_argument("--tag", default=None)
    args = ap.parse_args()

    tag = args.tag or f"dinov2s_{args.img_size}_g{args.grid}"
    shard_dir = args.root / "data" / "shards"
    out_dir = args.root / "data" / "embeddings" / tag
    out_dir.mkdir(parents=True, exist_ok=True)

    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",
                        handlers=[logging.FileHandler(out_dir / "embed.log", encoding="utf-8"),
                                  logging.StreamHandler()])
    log = logging.getLogger("embed")

    shards = sorted(shard_dir.glob("*.npz"))
    if not shards:
        raise SystemExit(f"no shards in {shard_dir}")
    todo = [p for p in shards if not (out_dir / p.name).exists()]
    if args.check:
        todo = todo[:3] if len(todo) >= 3 else shards[:3]
    elif args.limit:
        todo = todo[:args.limit]
    n_done = sum((out_dir / p.name).exists() for p in shards)
    log.info("%d shards, %d already embedded, %d to do -> %s", len(shards), n_done, len(todo), out_dir)

    emb = Embedder(args.model, args.img_size, args.grid, args.weights, batch_size=args.batch_size)
    cfg_path = out_dir / "config.json"
    if cfg_path.exists():
        old = json.loads(cfg_path.read_text())
        keys = ("model", "img_size", "grid", "mean", "std")
        if any(old.get(k) != emb.config.get(k) for k in keys):
            raise SystemExit(f"{cfg_path} was made with different settings; use another --tag")
    else:
        cfg_path.write_text(json.dumps(emb.config, indent=2))
    log.info("model %s on %s, img %d, grid %d, timm %s, torch %s", args.model, emb.device,
             args.img_size, args.grid, emb.config["timm"], emb.config["torch"])

    index_path = out_dir / "index.csv"
    new_index = not index_path.exists()
    t0, n_img, n_ok, n_err = time.time(), 0, 0, 0
    kept = {}
    with open(index_path, "a", encoding="utf-8", newline="") as idx:
        writer = csv.writer(idx)
        if new_index:
            writer.writerow(["StudyInstanceUID", "keys", "n_images", "error"])
        for i, (uid, images, segments, err) in enumerate(iterate(ShardDataset(todo), args.workers), 1):
            if not err:
                try:
                    feat, grid = emb(images)
                    out = split_outputs(feat, grid, segments)
                    save_atomic(out_dir / f"{uid}.npz", out)
                    if args.check:
                        kept[uid] = out
                    n_img += len(images)
                    n_ok += 1
                except Exception as e:  # noqa: BLE001
                    err = f"{type(e).__name__}: {e}"[:300]
            if err:
                n_err += 1
                log.warning("%s: %s", uid, err)
            keys = "|".join(k for k, *_ in segments) if not err else ""
            writer.writerow([uid, keys, 0 if err else len(images), err])
            idx.flush()
            if i % 100 == 0 or i == len(todo):
                el = time.time() - t0
                log.info("%d/%d  ok=%d err=%d  %.0f img/s  %.2f s/study  ETA %.1f min",
                         i, len(todo), n_ok, n_err, n_img / el, el / i, el / i * (len(todo) - i) / 60)

    if args.check and kept:
        log.info("sanity checks:")
        good = sanity_report(kept, log)
        el = time.time() - t0
        remaining = len([p for p in shards if not (out_dir / p.name).exists()])
        log.info("throughput %.0f images/s (includes model warm-up); %d studies left, "
                 "rough full-run time %.0f min", n_img / el, remaining,
                 el / max(n_ok, 1) * remaining / 60)
        log.info("CHECK %s", "PASSED" if good else "FAILED - send the log above")
    else:
        log.info("finished: ok=%d err=%d (failed studies are retried on the next run)", n_ok, n_err)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/src/models/folds.py
"""5-fold CV split over all training studies, grouped by patient.

    python -m src.models.folds

* Patient grouping: PatientID from data/raw/study_meta_chunk*.csv; all studies of a
  patient go to the same fold. Prints how many patients have more than one study.
* Stratification (greedy, deterministic): the 12 labels (soft label >= 0.5 counts as
  positive) plus the report language, so every fold has a similar mix of findings and
  reporting sites. The 58 human-labelled studies are spread across folds too.

Output: data/labels/folds.csv  StudyInstanceUID, PatientID, fold, lang, source
"""
from __future__ import annotations

import argparse
import re
from pathlib import Path

import numpy as np
import pandas as pd

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]

_LANG_HINTS = {
    "es": r"\b(menisco|derrame|rotura|ligamento|rodilla|sin|con|del)\b",
    "de": r"\b(meniskus|kreuzband|erguss|kniegelenk|und|der|die|kein)\b",
    "hr": r"\b(menisk[a-z]*|ligament[a-z]*|zglob[a-z]*|uredn[a-z]*|bez|uz)\b|[čćđ]",
    "nl": r"\b(van|het|een|geen|gewricht[a-z]*|kniegewricht)\b",
    "fr": r"\b(ménisque|épanchement|genou|du|des|pas)\b",
    "tr": r"[ığş]|\b(menisküs|bağ|izlendi|mevcuttur|normaldir)\b",
}


def guess_language(text: str) -> str:
    """Rough report-language guess (used only to balance folds)."""
    t = str(text).lower()
    if re.search(r"[а-яё]", t):
        return "bg"
    if re.search(r"[α-ω]", t):
        return "el"
    scores = {lang: len(re.findall(pat, t)) for lang, pat in _LANG_HINTS.items()}
    lang, best = max(scores.items(), key=lambda kv: kv[1])
    en = len(re.findall(r"\b(the|of|and|is|with|no|meniscus|ligament|effusion)\b", t))
    return lang if best > max(2, en) else "en"


def read_labels(path: Path) -> pd.DataFrame:
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path, dtype={"StudyInstanceUID": str})


def load_patients(raw: Path) -> pd.Series | None:
    files = sorted(raw.glob("study_meta_chunk*.csv"))
    if not files:
        return None
    meta = pd.concat([pd.read_csv(f, usecols=["StudyInstanceUID", "PatientID"], dtype=str)
                      for f in files])
    return meta.drop_duplicates("StudyInstanceUID").set_index("StudyInstanceUID")["PatientID"]


def greedy_group_folds(Y: np.ndarray, groups: np.ndarray, k: int, seed: int) -> np.ndarray:
    """Assign each group to a fold, keeping every column's positives and the fold
    sizes as even as possible. Rare columns are placed first."""
    G = groups.max() + 1
    gY = np.zeros((G, Y.shape[1]))
    np.add.at(gY, groups, Y)
    gsize = np.bincount(groups, minlength=G).astype(float)
    totals = Y.sum(0)
    tgt = totals / k
    tgt_size = len(groups) / k
    w = 1.0 / np.maximum(tgt, 1.0)

    rng = np.random.default_rng(seed)
    with np.errstate(divide="ignore"):
        rarity = np.where(gY > 0, totals[None, :], np.inf).min(1)   # rarest column present
    order = np.lexsort((rng.random(G), -gsize, rarity))

    cnt = np.zeros((k, Y.shape[1]))
    size = np.zeros(k)
    fold_of_group = np.empty(G, dtype=int)
    for g in order:
        before = (w * (cnt - tgt) ** 2).sum(1) + ((size - tgt_size) / tgt_size) ** 2 * 50
        after = (w * (cnt + gY[g] - tgt) ** 2).sum(1) + ((size + gsize[g] - tgt_size) / tgt_size) ** 2 * 50
        f = int(np.argmin(after - before))
        fold_of_group[g] = f
        cnt[f] += gY[g]
        size[f] += gsize[g]
    return fold_of_group[groups]


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", type=Path, default=Path("."))
    ap.add_argument("--labels", type=Path, default=None)
    ap.add_argument("--k", type=int, default=5)
    ap.add_argument("--seed", type=int, default=2026)
    args = ap.parse_args()

    raw = args.root / "data" / "raw"
    lab_path = args.labels or args.root / "data" / "labels" / "soft_labels.parquet"
    labels = read_labels(lab_path)
    train = pd.read_csv(raw / "train.csv", usecols=["StudyInstanceUID", "Report"])
    df = labels.merge(train, on="StudyInstanceUID", how="left")
    df = df[df["source"] != "missing"].reset_index(drop=True)

    pids = load_patients(raw)
    if pids is None:
        print("WARNING: no study_meta_chunk*.csv in data/raw -> no patient grouping")
        df["PatientID"] = ""
    else:
        df["PatientID"] = df["StudyInstanceUID"].map(pids).fillna("")
    key = df["PatientID"].where(df["PatientID"] != "", "study:" + df["StudyInstanceUID"])
    groups = pd.factorize(key)[0]
    per_patient = pd.Series(groups).value_counts()
    print(f"studies {len(df)} · patients {groups.max() + 1} · "
          f"patients with >1 study: {(per_patient > 1).sum()} "
          f"(covering {per_patient[per_patient > 1].sum()} studies) · "
          f"missing PatientID: {(df['PatientID'] == '').sum()}")

    df["lang"] = df["Report"].map(guess_language)
    Yl = (df[LABELS].to_numpy() >= 0.5).astype(float)
    Ylang = pd.get_dummies(df["lang"]).to_numpy().astype(float)
    Yhuman = (df["source"] == "human").to_numpy()[:, None].astype(float)
    Y = np.hstack([Yl, Ylang, Yhuman])
    df["fold"] = greedy_group_folds(Y, groups, args.k, args.seed)

    out = args.root / "data" / "labels" / "folds.csv"
    df[["StudyInstanceUID", "PatientID", "fold", "lang", "source"]].to_csv(out, index=False)

    summary = pd.DataFrame(Yl, columns=LABELS).groupby(df["fold"]).sum().astype(int)
    summary.insert(0, "studies", df.groupby("fold").size())
    summary["human"] = df.groupby("fold")["source"].apply(lambda s: (s == "human").sum())
    with pd.option_context("display.width", 200, "display.max_columns", 30):
        print("\npositives (>= 0.5) per fold:")
        print(summary.to_string())
        print("\nlanguages per fold:")
        print(pd.crosstab(df["fold"], df["lang"]).to_string())
    if pids is not None:
        leak = df.groupby("PatientID")["fold"].nunique()
        leak = leak[(leak > 1) & (leak.index != "")]
        print(f"\npatients split across folds: {len(leak)} (must be 0)")
    print(f"\nsaved -> {out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/src/models/metrics.py
"""Out-of-fold scoring for the 12 labels.

Two views, reported side by side:
  auc_llm   : LLM-labelled studies; per label the 'uncertain' (0.5) rows are left out
              and the rest are 1/0. Main signal (thousands of studies).
  auc_human : the 58 human-labelled studies (true labels). Small and noisy, but the
              only check against the organisers' own definitions.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
          "Contusion", "Fracture"]


def safe_auc(y: np.ndarray, s: np.ndarray) -> float:
    y = np.asarray(y)
    if len(y) == 0 or y.min() == y.max():
        return float("nan")
    return float(roc_auc_score(y, s))


def score(pred: np.ndarray, target: np.ndarray, source: np.ndarray) -> pd.DataFrame:
    """pred, target: [n, 12]; source: 'llm' / 'human' per row."""
    rows = []
    llm = source == "llm"
    hum = source == "human"
    for j, lab in enumerate(LABELS):
        t, p = target[:, j], pred[:, j]
        certain = llm & (t != 0.5)
        rows.append(dict(
            label=lab,
            auc_llm=safe_auc(t[certain] >= 0.5, p[certain]),
            pos_llm=int((t[certain] >= 0.5).sum()),
            auc_human=safe_auc(t[hum] >= 0.5, p[hum]),
            pos_human=int((t[hum] >= 0.5).sum()),
        ))
    df = pd.DataFrame(rows).set_index("label")
    df.loc["MACRO"] = [df["auc_llm"].mean(), df["pos_llm"].sum(),
                       df["auc_human"].mean(), df["pos_human"].sum()]
    return df


In [ ]:
%%writefile /kaggle/working/src/models/m1.py
"""M1: frozen DINOv2-S slice features -> small transformer aggregator -> 12 labels.

    python -m src.models.m1 --smoke          # 1 fold, 2 epochs, 400 studies: pipeline check
    python -m src.models.m1 --name base      # 5-fold CV, OOF scores, checkpoints

Needs: data/labels/soft_labels.parquet, data/labels/folds.csv (src.models.folds) and
the feature cache (src.models.data). Outputs go to runs/m1/<name>/:
    fold{k}.pt, oof.csv, metrics.csv, config.json, train.log

Per study the input is 4 series kinds x 32 slices = 128 tokens; each token is the
slice's CLS + mean-patch feature (768) plus, by default, its 2x2 patch grid (1536).
token = MLP(feature) + kind embedding + slice-position embedding
-> 2-layer transformer -> 12 learned label queries attend over the tokens -> 12 logits.

Inputs are standardised per channel (stats.npz in the cache dir, also saved in every
checkpoint). A masked mean of the tokens is added to each label's attention output, so
the model learns from the first epoch instead of waiting for the attention to focus.

Training: BCE on soft labels (1 / 0.5 / 0), AdamW + cosine, and augmentation:
  * mirror the knee left<->right (p=0.5): coronal/axial use the flipped-image
    features, sagittal kinds get their slice order reversed;
  * series dropout (drop whole kinds, keep >= 1) and slice dropout.
Validation/prediction average the normal and mirrored knee (TTA).
"""
from __future__ import annotations

import argparse
import json
import logging
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from .data import KIND_SLOT, KIND_SLOT_MIRROR, N_SLICES, SAGITTAL, load_cache, load_stats
from .folds import read_labels
from .metrics import LABELS, score

N_KINDS = 4


# ----------------------------------------------------------------------------- model
class M1(nn.Module):
    def __init__(self, in_dim: int, d: int = 256, layers: int = 2, heads: int = 4,
                 drop: float = 0.1, n_labels: int = len(LABELS)):
        super().__init__()
        self.inp = nn.Sequential(nn.LayerNorm(in_dim), nn.Linear(in_dim, d), nn.GELU(),
                                 nn.Dropout(drop), nn.Linear(d, d))
        self.kind = nn.Parameter(torch.randn(N_KINDS, 1, d) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, N_SLICES, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, drop, activation="gelu",
                                           batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.queries = nn.Parameter(torch.randn(n_labels, d) * 0.02)
        self.pool = nn.MultiheadAttention(d, heads, dropout=drop, batch_first=True)
        self.norm = nn.LayerNorm(d)
        self.head_w = nn.Parameter(torch.zeros(n_labels, d))
        self.head_b = nn.Parameter(torch.zeros(n_labels))

    def forward(self, x: torch.Tensor, keep: torch.Tensor) -> torch.Tensor:
        """x [B, K, S, F] float; keep [B, K, S] bool (True = real token)."""
        b, k, s, _ = x.shape
        h = self.inp(x) + self.kind[None] + self.pos[None]          # [B,K,S,d]
        h = h.reshape(b, k * s, -1)
        pad = ~keep.reshape(b, k * s)
        h = self.enc(h, src_key_padding_mask=pad)
        w = keep.reshape(b, k * s, 1).to(h.dtype)
        h_mean = (h * w).sum(1) / w.sum(1).clamp(min=1.0)                # [B,d]
        q = self.queries[None].expand(b, -1, -1)
        o, _ = self.pool(q, h, h, key_padding_mask=pad)
        o = self.norm(o + h_mean[:, None])     # mean path: learns fast; attention refines
        return (o * self.head_w).sum(-1) + self.head_b                 # [B,12]


# ----------------------------------------------------------------------------- batches
class Features:
    """Holds the cache on the CPU and assembles batches (with optional mirroring)."""

    def __init__(self, cache: dict, use_grid: bool, stats: dict):
        self.stats = stats
        self.f_mean = torch.from_numpy(stats["f_mean"])
        self.f_std = torch.from_numpy(stats["f_std"])
        self.g_mean = torch.from_numpy(stats["g_mean"])
        self.g_std = torch.from_numpy(stats["g_std"])
        self.feats = torch.from_numpy(np.ascontiguousarray(cache["feats"]))
        self.grid = torch.from_numpy(np.ascontiguousarray(cache["grid"])) if use_grid else None
        self.present = torch.from_numpy(cache["present"])
        self.slot = torch.from_numpy(KIND_SLOT)
        self.slot_m = torch.from_numpy(KIND_SLOT_MIRROR)
        self.sag = torch.from_numpy(SAGITTAL)
        self.in_dim = self.feats.shape[-1] + (self.grid.shape[-2] * self.grid.shape[-1]
                                              if use_grid else 0)

    def batch(self, rows: torch.Tensor, mirror: torch.Tensor):
        """rows [B] cache rows, mirror [B] bool -> x [B,4,32,F] float32, present [B,4] bool."""
        slots = torch.where(mirror[:, None], self.slot_m[None], self.slot[None])   # [B,4]
        r = rows[:, None].expand(-1, N_KINDS)
        x = (self.feats[r, slots].float() - self.f_mean) / self.f_std
        if self.grid is not None:
            g = (self.grid[r, slots].float() - self.g_mean) / self.g_std     # [B,4,32,c,384]
            x = torch.cat([x, g.flatten(-2)], -1)
        present = self.present[r, slots]
        flip = (mirror[:, None] & self.sag[None])[:, :, None, None]
        x = torch.where(flip, x.flip(2), x)
        return x, present


def token_mask(present: torch.Tensor, train: bool, p_series: float, p_slice: float) -> torch.Tensor:
    """[B,4] kinds present -> [B,4,32] tokens kept (dropout only when training)."""
    kinds = present.clone()
    if train and p_series > 0:
        drop = (torch.rand(kinds.shape) < p_series) & kinds
        survivors = kinds & ~drop
        drop[~survivors.any(1)] = False            # never drop every series
        kinds = kinds & ~drop
    keep = kinds[:, :, None].expand(-1, -1, N_SLICES).clone()
    keep[~kinds.any(1), 2, 0] = True               # no series at all: one dummy token
    if train and p_slice > 0:
        keep &= torch.rand(keep.shape) >= p_slice
        empty = ~keep.flatten(1).any(1)
        keep[empty] = kinds[empty][:, :, None].expand(-1, -1, N_SLICES)
    return keep


@torch.no_grad()
def predict(model: M1, feats: Features, rows: torch.Tensor, device, tta: bool = True,
            bs: int = 256) -> np.ndarray:
    model.eval()
    out = []
    for s in range(0, len(rows), bs):
        r = rows[s:s + bs]
        acc = 0
        views = (False, True) if tta else (False,)
        for m in views:
            x, present = feats.batch(r, torch.full((len(r),), m))
            keep = token_mask(present, False, 0, 0)
            acc = acc + torch.sigmoid(model(x.to(device), keep.to(device))).cpu()
        out.append(acc / len(views))
    return torch.cat(out).numpy()


# ----------------------------------------------------------------------------- training
def lr_lambda(step: int, total: int, warmup: int) -> float:
    if step < warmup:
        return (step + 1) / warmup
    t = (step - warmup) / max(1, total - warmup)
    return 0.5 * (1 + math.cos(math.pi * min(t, 1.0)))


def train_fold(k: int, df: pd.DataFrame, Y: torch.Tensor, rows: torch.Tensor,
               feats: Features, args, device, log) -> tuple[np.ndarray, M1]:
    torch.manual_seed(args.seed + k)
    tr = torch.from_numpy(np.flatnonzero(df["fold"].to_numpy() != k))
    va = torch.from_numpy(np.flatnonzero(df["fold"].to_numpy() == k))
    model = M1(feats.in_dim, args.d, args.layers, args.heads, args.drop).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)
    steps_per_epoch = math.ceil(len(tr) / args.bs)
    total = steps_per_epoch * args.epochs
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda st: lr_lambda(st, total, steps_per_epoch))
    src = df["source"].to_numpy()
    for ep in range(1, args.epochs + 1):
        model.train()
        t0, run_loss = time.time(), 0.0
        perm = tr[torch.randperm(len(tr))]
        for s in range(0, len(perm), args.bs):
            idx = perm[s:s + args.bs]
            mirror = torch.rand(len(idx)) < args.p_mirror
            x, present = feats.batch(rows[idx], mirror)
            keep = token_mask(present, True, args.p_series, args.p_slice)
            logits = model(x.to(device), keep.to(device))
            loss = F.binary_cross_entropy_with_logits(logits, Y[idx].to(device))
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            run_loss += loss.item() * len(idx)
        if ep % args.eval_every == 0 or ep == args.epochs:
            p = predict(model, feats, rows[va], device, tta=True)
            sc = score(p, Y[va].numpy(), src[va.numpy()])
            log.info("fold %d ep %2d  loss %.4f  val auc_llm %.4f  auc_human %.3f  (%.1fs)",
                     k, ep, run_loss / len(tr), sc.loc["MACRO", "auc_llm"],
                     sc.loc["MACRO", "auc_human"], time.time() - t0)
    return predict(model, feats, rows[va], device, tta=True), model


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", type=Path, default=Path("."))
    ap.add_argument("--tag", default="dinov2s_224_g4")
    ap.add_argument("--cache", default="cache_g2")
    ap.add_argument("--name", default=None)
    ap.add_argument("--folds", default="0,1,2,3,4")
    ap.add_argument("--no-grid", action="store_true", help="use only CLS+mean features")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--bs", type=int, default=64)
    ap.add_argument("--lr", type=float, default=5e-4)
    ap.add_argument("--wd", type=float, default=0.05)
    ap.add_argument("--d", type=int, default=256)
    ap.add_argument("--layers", type=int, default=2)
    ap.add_argument("--heads", type=int, default=4)
    ap.add_argument("--drop", type=float, default=0.1)
    ap.add_argument("--p-mirror", type=float, default=0.5)
    ap.add_argument("--p-series", type=float, default=0.2)
    ap.add_argument("--p-slice", type=float, default=0.1)
    ap.add_argument("--eval-every", type=int, default=5)
    ap.add_argument("--seed", type=int, default=2026)
    ap.add_argument("--smoke", action="store_true")
    args = ap.parse_args()
    if args.smoke:
        args.folds, args.epochs, args.eval_every = "0", 2, 1
        args.name = args.name or "smoke"

    name = args.name or time.strftime("%Y%m%d_%H%M")
    out = args.root / "runs" / "m1" / name
    out.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s",
                        handlers=[logging.FileHandler(out / "train.log", encoding="utf-8"),
                                  logging.StreamHandler()], force=True)
    log = logging.getLogger("m1")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    labels = read_labels(args.root / "data" / "labels" / "soft_labels.parquet")
    folds = pd.read_csv(args.root / "data" / "labels" / "folds.csv", dtype={"StudyInstanceUID": str})
    cache_dir = args.root / "data" / "embeddings" / args.tag / args.cache
    cache = load_cache(cache_dir)
    stats = load_stats(cache_dir, cache)
    row_of = {u: i for i, u in enumerate(cache["uids"])}
    df = folds.merge(labels[["StudyInstanceUID", *LABELS]], on="StudyInstanceUID")
    df["row"] = df["StudyInstanceUID"].map(row_of)
    missing = df["row"].isna().sum()
    if missing:
        log.warning("%d studies have no embeddings and are skipped", missing)
    df = df.dropna(subset=["row"]).reset_index(drop=True)
    if args.smoke:
        df = df.groupby("fold").head(80).reset_index(drop=True)

    feats = Features(cache, use_grid=not args.no_grid, stats=stats)
    Y = torch.tensor(df[LABELS].to_numpy(), dtype=torch.float32)
    rows = torch.tensor(df["row"].astype(int).to_numpy())
    cfg = {k: (str(v) if isinstance(v, Path) else v) for k, v in vars(args).items()}
    cfg.update(in_dim=feats.in_dim, n_studies=len(df), device=str(device),
               torch=torch.__version__)
    (out / "config.json").write_text(json.dumps(cfg, indent=2))
    log.info("M1 %s: %d studies, in_dim %d, device %s", name, len(df), feats.in_dim, device)

    oof = pd.DataFrame(index=df.index, columns=LABELS, dtype=float)
    for k in [int(f) for f in args.folds.split(",")]:
        pred, model = train_fold(k, df, Y, rows, feats, args, device, log)
        oof.loc[df.index[df["fold"] == k], LABELS] = pred
        torch.save({"state_dict": model.state_dict(), "config": cfg,
                    "stats": {k2: torch.from_numpy(v) for k2, v in stats.items()}},
                   out / f"fold{k}.pt")

    done = oof.notna().all(axis=1).to_numpy()
    res = df.loc[done, ["StudyInstanceUID", "fold", "source"]].join(oof.loc[done])
    res.to_csv(out / "oof.csv", index=False)
    sc = score(oof.loc[done].to_numpy(), Y.numpy()[done], df["source"].to_numpy()[done])
    sc.to_csv(out / "metrics.csv")
    with pd.option_context("display.float_format", "{:.4f}".format):
        log.info("\nOOF scores (%d studies):\n%s", done.sum(), sc.to_string())
    log.info("saved -> %s", out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/src/models/predict.py
"""Inference: study DICOMs -> 12 probabilities. Shared by the Kaggle submission notebook
and a local parity check.

Local check (laptop, before uploading anything):
    python -m src.models.predict --check --run v2
  re-embeds a few training shards with the same Embedder, compares the features with the
  training cache, and checks the fold ensemble gives the same predictions from both.

Kaggle (called from the submission notebook):
    run_test(data_root, model_dir, "/kaggle/working/submission.csv")
  DICOM -> kneeprep (CPU worker pool) -> DINOv2 features (GPU) -> M1 fold ensemble with
  mirror TTA -> submission.csv in sample_submission's row and column order. A study that
  fails at any step gets the training prevalence instead, so the run never crashes.
"""
from __future__ import annotations

import argparse
import json
import os
import time
from pathlib import Path

import numpy as np

from .data import SLOTS, pool_grid
from .embed import Embedder, plan_images, split_outputs
from .metrics import LABELS

N_SLOTS, N_SLICES, FEAT_DIM, PATCH_DIM = len(SLOTS), 32, 768, 384
SERIES_COLS = ["SeriesInstanceUID", "Fluid_Sensitive", "Fat_Suppression", "Anatomical_Plane"]


# ----------------------------------------------------------------------------- features
def study_tensors(arrays: dict, embedder: Embedder, g_out: int = 2):
    """kneeprep arrays -> (feats [6,32,768] f16, grid [6,32,g*g,384] f16, present [6] bool)."""
    feats = np.zeros((N_SLOTS, N_SLICES, FEAT_DIM), np.float16)
    grid = np.zeros((N_SLOTS, N_SLICES, g_out * g_out, PATCH_DIM), np.float16)
    present = np.zeros(N_SLOTS, bool)
    images, segments = plan_images(arrays)
    if not segments:
        return feats, grid, present
    f, g = embedder(images)
    out = split_outputs(f, g, segments)
    for j, slot in enumerate(SLOTS):
        if slot in out:
            feats[j] = out[slot]
            grid[j] = pool_grid(out[f"{slot}_grid"], g_out)
            present[j] = True
    return feats, grid, present


# ----------------------------------------------------------------------------- model ensemble
class Ensemble:
    """All fold checkpoints of one M1 run; averages their (mirror-TTA) probabilities."""

    def __init__(self, ckpt_paths, device=None):
        import torch

        from .m1 import M1
        self.torch = torch
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.models, self.cfg, self.stats = [], None, None
        for p in sorted(ckpt_paths):
            ck = torch.load(p, map_location="cpu", weights_only=False)
            cfg = ck["config"]
            m = M1(cfg["in_dim"], cfg["d"], cfg["layers"], cfg["heads"], cfg["drop"])
            m.load_state_dict(ck["state_dict"])
            self.models.append(m.eval().to(self.device))
            if self.cfg is None:
                self.cfg = cfg
                self.stats = {k: v.numpy() for k, v in ck["stats"].items()}
        if not self.models:
            raise FileNotFoundError("no fold*.pt checkpoints given")
        self.use_grid = not self.cfg.get("no_grid", False)

    def predict(self, feats: np.ndarray, grid: np.ndarray, present: np.ndarray) -> np.ndarray:
        from .m1 import Features, predict
        torch = self.torch
        fs = Features(dict(feats=feats, grid=grid, present=present), self.use_grid, self.stats)
        rows = torch.arange(len(feats))
        preds = [predict(m, fs, rows, self.device, tta=True) for m in self.models]
        return np.mean(preds, 0)


# ----------------------------------------------------------------------------- Kaggle run
def prep_task(task):
    """Pool worker: (index, study_dir, series rows) -> (index, arrays | None, message)."""
    i, study_dir, rows = task
    try:
        from src.kneeprep import process_study
        if not rows:
            return i, None, "no rows in test_series.csv"
        if not os.path.isdir(study_dir):
            return i, None, "study folder missing"
        arrays, series_meta, _ = process_study(study_dir, rows)
        failed = sum(int(m.get("n_failed", 0) or 0) for m in series_meta)
        errs = [m["error"] for m in series_meta if m.get("error")]
        msg = f"failed slice reads: {failed}" if failed else ""
        if errs:
            msg += f" series errors: {errs[:2]}"
        return i, (arrays or None), msg
    except Exception as e:  # noqa: BLE001
        return i, None, f"{type(e).__name__}: {e}"[:300]


def run_test(data_root, model_dir, out_csv, workers: int | None = None,
             time_limit_h: float = 8.0, split: str = "test", log=print) -> "pd.DataFrame":
    import multiprocessing as mp

    import pandas as pd

    t0 = time.time()
    data_root, model_dir = Path(data_root), Path(model_dir)
    series = pd.read_csv(data_root / f"{split}_series.csv", dtype={"StudyInstanceUID": str,
                                                                   "SeriesInstanceUID": str})
    sample = pd.read_csv(data_root / "sample_submission.csv", dtype={"StudyInstanceUID": str})
    groups = {s: g[SERIES_COLS].to_dict("records") for s, g in series.groupby("StudyInstanceUID")}
    ids = sample["StudyInstanceUID"].tolist()
    priors = json.loads(next(model_dir.rglob("priors.json")).read_text())
    log(f"{len(ids)} studies to predict | {len(groups)} studies in {split}_series.csv")

    # Start the CPU pool BEFORE touching CUDA (fork after CUDA init is unsafe).
    workers = workers or os.cpu_count() or 2
    ctx = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else mp.get_context()
    pool = ctx.Pool(workers)
    tasks = [(i, str(data_root / f"{split}_series" / s), groups.get(s, [])) for i, s in enumerate(ids)]
    results = pool.imap_unordered(prep_task, tasks)

    weights = next(model_dir.rglob("model.safetensors"), None)
    if weights is None:
        log("WARNING: no model.safetensors in the model dataset - timm will try to download")
    embedder = Embedder(weights=str(weights) if weights else None)
    ens = Ensemble(list(model_dir.rglob("fold*.pt")), embedder.device)
    g_out = int(ens.cfg.get("cache", "cache_g2").split("_g")[-1])
    log(f"models ready: {len(ens.models)} folds, grid {g_out}, device {embedder.device} "
        f"({time.time() - t0:.0f}s)")

    n = len(ids)
    feats = np.zeros((n, N_SLOTS, N_SLICES, FEAT_DIM), np.float16)
    grid = np.zeros((n, N_SLOTS, N_SLICES, g_out * g_out, PATCH_DIM), np.float16)
    present = np.zeros((n, N_SLOTS), bool)
    problems, timed_out = {}, False
    for k, (i, arrays, msg) in enumerate(results, 1):
        if msg:
            problems[ids[i]] = msg
        if arrays:
            try:
                feats[i], grid[i], present[i] = study_tensors(arrays, embedder, g_out)
            except Exception as e:  # noqa: BLE001
                problems[ids[i]] = f"embed {type(e).__name__}: {e}"[:300]
        el = time.time() - t0
        if k % 50 == 0 or k == n:
            log(f"  {k}/{n} studies  {el / k:.2f} s/study  elapsed {el / 60:.1f} min  "
                f"ETA {el / k * (n - k) / 60:.1f} min")
        if el > time_limit_h * 3600:
            log(f"TIME LIMIT reached after {k} studies - the rest get prior values")
            timed_out = True
            break
    pool.terminate()

    ok = present.any(1)
    pred = np.tile(np.array([priors[lab] for lab in LABELS], float), (n, 1))
    if ok.any():
        pred[ok] = ens.predict(feats[ok], grid[ok], present[ok])
    sub = pd.DataFrame(pred, columns=LABELS)
    sub.insert(0, "StudyInstanceUID", ids)
    sub = sub[sample.columns.tolist()]
    sub.to_csv(out_csv, index=False)

    log(f"\npredicted {ok.sum()}/{n} studies from images, {n - ok.sum()} got priors"
        f"{' (time limit)' if timed_out else ''} | total {(time.time() - t0) / 60:.1f} min")
    log("series kinds present: " + str(dict(zip(SLOTS, present.sum(0).tolist()))))
    for sid, msg in list(problems.items())[:20]:
        log(f"  note {sid[-12:]}: {msg}")
    if len(problems) > 20:
        log(f"  ... {len(problems) - 20} more notes")
    log(f"saved -> {out_csv}")
    return sub


# ----------------------------------------------------------------------------- local check
def local_check(root: Path, run: str, tag: str, cache_name: str, n: int) -> None:
    from .data import load_cache
    run_dir = root / "runs" / "m1" / run
    cache = load_cache(root / "data" / "embeddings" / tag / cache_name, in_memory=False)
    uids = cache["uids"][:n]
    embedder = Embedder()
    ens = Ensemble(list(run_dir.glob("fold*.pt")), embedder.device)
    g_out = int(cache_name.split("_g")[-1])

    t0 = time.time()
    F = np.zeros((n, N_SLOTS, N_SLICES, FEAT_DIM), np.float16)
    G = np.zeros((n, N_SLOTS, N_SLICES, g_out * g_out, PATCH_DIM), np.float16)
    P = np.zeros((n, N_SLOTS), bool)
    for i, u in enumerate(uids):
        with np.load(root / "data" / "shards" / f"{u}.npz") as z:
            arrays = {k: z[k] for k in z.files}
        F[i], G[i], P[i] = study_tensors(arrays, embedder, g_out)
    print(f"re-embedded {n} studies in {time.time() - t0:.1f}s")

    cF = np.asarray(cache["feats"][:n]).astype(np.float32)
    cP = np.asarray(cache["present"][:n])
    a, b = F.astype(np.float32)[P], cF[cP]
    cos = (a * b).sum(-1) / (np.linalg.norm(a, axis=-1) * np.linalg.norm(b, axis=-1) + 1e-8)
    print(f"present slots identical: {np.array_equal(P, cP)}")
    print(f"feature cosine vs training cache: min {cos.min():.5f}, mean {cos.mean():.5f}   "
          f"[expect > 0.999]")
    p_new = ens.predict(F, G, P)
    p_old = ens.predict(np.asarray(cache["feats"][:n]), np.asarray(cache["grid"][:n]), cP)
    diff = np.abs(p_new - p_old).max()
    print(f"ensemble ({len(ens.models)} folds) max |prediction difference|: {diff:.5f}   "
          f"[expect < 0.01]")
    ok = np.array_equal(P, cP) and cos.min() > 0.999 and diff < 0.01
    print("PARITY CHECK", "PASSED" if ok else "FAILED - send this output")


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", type=Path, default=Path("."))
    ap.add_argument("--check", action="store_true")
    ap.add_argument("--run", default="v2")
    ap.add_argument("--tag", default="dinov2s_224_g4")
    ap.add_argument("--cache", default="cache_g2")
    ap.add_argument("--n", type=int, default=16)
    args = ap.parse_args()
    if not args.check:
        raise SystemExit("use --check locally; run_test() is called from the Kaggle notebook")
    local_check(args.root, args.run, args.tag, args.cache, args.n)


if __name__ == "__main__":
    main()


## Predict the test set

In [ ]:
sys.path.insert(0, "/kaggle/working")
from src.models.predict import run_test

remaining_h = TIME_LIMIT_H - (time.time() - T_START) / 3600
sub = run_test(DATA_ROOT, MODEL_DIR, "/kaggle/working/submission.csv",
               workers=WORKERS, time_limit_h=remaining_h)
print(sub.shape)
print(sub.head())
print(sub.drop(columns="StudyInstanceUID").describe().T[["mean", "std", "min", "max"]])